In [0]:
import re
from datetime import datetime

def archive_raw_files(raw_base_path: str, archive_base_path: str):

    try:
        folders = dbutils.fs.ls(raw_base_path)

        for folder in folders:

            if not folder.isDir():
                continue

            table_name = folder.name.rstrip("/")
            print(f"\nProcessing folder: {table_name}")

            files = dbutils.fs.ls(folder.path)

            file_dates = {}
            skipped = []

            # Extract dates from all files
            for file in files:

                if file.isDir():
                    continue

                # Extract timestamp from filename
                
                match = re.search(r'_(\d{8})\d{6}\.csv$', file.name)

                if not match:
                    skipped.append(file.name)
                    continue

                date_str = match.group(1)

                try:
                    file_date = datetime.strptime(date_str, "%d%m%Y").date()
                    
                    if file_date not in file_dates:
                        file_dates[file_date] = []
                    file_dates[file_date].append(file)

                except:
                    skipped.append(file.name)

            # logs
            if skipped:
                print(f"Skipped: {skipped}")

            if not file_dates:
                print("No valid files found → skipping")
                continue

            # Find the MOST RECENT date
            most_recent_date = max(file_dates.keys())
            print(f"Most recent date: {most_recent_date}")

           
            files_to_keep = file_dates[most_recent_date]
            files_to_archive = []
            
            for date, files_list in file_dates.items():
                if date != most_recent_date:
                    files_to_archive.extend(files_list)

            print(f"Files to keep (most recent): {len(files_to_keep)}")
            print(f"Files to archive (older): {len(files_to_archive)}")

            if not files_to_archive:
                print("No old files to archive")
                continue

            timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            archive_dir = f"{archive_base_path}{table_name}/{timestamp}/"

            print(f"Archiving {len(files_to_archive)} files from {table_name}")

            for file in files_to_archive:
                src = file.path
                dest = f"{archive_dir}{file.name}"

                print(f"Moving {file.name}")
                dbutils.fs.mv(src, dest)

            print(f"Archived {len(files_to_archive)} files → {archive_dir}")

    except Exception as e:
        print(f"Error: {str(e)}")



S3_BUCKET = "s3a://retail-etl-lakehouse-gopika"

raw_path = f"{S3_BUCKET}/raw/"
archive_path = f"{S3_BUCKET}/archive/"

print("Starting RAW archival process...")
archive_raw_files(raw_path, archive_path)
print("Done.")

In [0]:
import re
from datetime import datetime

raw_path = "s3a://retail-etl-lakehouse-gopika/raw/"

print("Files remaining in /raw/ folder:\n")
print(f"{'Folder':<15} {'Filename':<50} {'Date in filename':<20}")
print("-" * 85)

folders = dbutils.fs.ls(raw_path)
for folder in folders:
    if folder.isDir():
        table_name = folder.name.rstrip("/")
        files = dbutils.fs.ls(folder.path)
        
        for file in files:
            if not file.isDir():
                match = re.search(r'_(\d{8})\d{6}\.csv$', file.name)
                if match:
                    date_str = match.group(1)
                    file_date = datetime.strptime(date_str, "%d%m%Y").strftime("%Y-%m-%d")
                    print(f"{table_name:<15} {file.name:<50} {file_date:<20}")
                else:
                    print(f"{table_name:<15} {file.name:<50} {'Unknown format':<20}")

In [0]:
archive_base = "s3a://retail-etl-lakehouse-gopika/archive/"

print("Files in ARCHIVE folder:\n")
print(f"{'Table Folder':<15} {'Archive Timestamp':<25} {'Filename':<50} {'Date in filename':<20}")
print("-" * 110)

import re
from datetime import datetime

table_folders = dbutils.fs.ls(archive_base)

for table_folder in table_folders:
    if table_folder.isDir():
        table_name = table_folder.name.rstrip("/")
        
        timestamp_folders = dbutils.fs.ls(table_folder.path)
        
        for ts_folder in timestamp_folders:
            if ts_folder.isDir():
                ts_name = ts_folder.name.rstrip("/")
                
                files = dbutils.fs.ls(ts_folder.path)
                
                for file in files:
                    if not file.isDir():
                        match = re.search(r'_(\d{8})\d{6}\.csv$', file.name)
                        if match:
                            date_str = match.group(1)
                            file_date = datetime.strptime(date_str, "%d%m%Y").strftime("%Y-%m-%d")
                            print(f"{table_name:<15} {ts_name:<25} {file.name:<50} {file_date:<20}")